# FX quote lift — logistic GD + sklearn (Practice Skeleton)

**Short name (GitHub):** `FX_Quote_GD_Sklearn`  
**Lab source:** Coursera C1_W3 Lab06 + Lab07, **adapted to the currency-conversion industry**  
**Language:** Python (NumPy + Matplotlib + scikit-learn)

A multi-currency conversion platform (retail travel money, SME invoices, corporate treasury) streams a live quote. The client either **lifts** the quote (converts) or **passes**. You will train a logistic model for $P(\text{lift})$ from two desk scores, then the same optimizer on spread and a USD/CRC markup cart.

Use this notebook to practice. Open **`FX_Quote_GD_Sklearn_Solution.ipynb`** only after you attempt each exercise.

Companion files: `FX_Quote_GD_Sklearn_Cheatsheet.docx`, `FX_Quote_GD_Sklearn_Reusable_Template.ipynb`, `fx_quote_gd_flowchart.png`, `FX_Quote_GD_Sklearn.py`.

### Learning objectives
- Implement loop + vectorized logistic gradients on an FX blotter
- Run batch GD and watch the fill-cost $J$ fall
- Draw the $P(\text{lift})=0.5$ boundary (pricing / sales view)
- Fit `sklearn.linear_model.LogisticRegression` and compare **C=∞ vs default L2**
- Extra practice: 60-quote stream + USD→CRC markup vs conversion
- Monte-Carlo of learning rate, blotter size, and last-look / stale-quote noise
- Rewrite the same fill-rate result for a pricing quant, a sales dealer, a treasury executive, and a retail converter

### Data files
- `data/fx_quote_blotter6.csv` — 6-row toy blotter (same numbers as Lab06, so the published gradient check still holds)
- `data/fx_spread_1d.csv` — quoted spread (bps) vs converted
- `data/fx_quote_practice.csv` — 60 streamed quotes
- `data/fx_usdcrc_markup.csv` — USD→CRC retail markup vs converted


## Inline cheat-sheet

See also **`FX_Quote_GD_Sklearn_Cheatsheet.docx`**.

| Item | FX reading / code |
|------|-------------------|
| $f=\sigma(w\cdot x+b)$ | Model **P(lift)** or P(convert) |
| $y=1$ | Client lifted the quote / completed the conversion |
| Cost $J$ | Average surprise of the quoted probability vs the blotter |
| $\partial J/\partial w=(1/m)X^\top(f-y)$ | How to nudge edge and urgency weights |
| Threshold $\tau$ | Sales risk appetite: lift if $f\ge\tau$ (0.5 in the lab) |
| Lab06 check | $w=(2,3),b=1$ → `dj_db≈0.49862`, `dj_dw≈[0.49833,0.49884]` |
| sklearn | `C=np.inf` ≈ unregularized; default `C=1` is L2 on the weights |
| Spread 1-D | Fit on **tightness_bps = 34 − spread_bps** so $w>0$ |

**Flow:** blotter → score $z$ → $P(\text{lift})$ → error $(f-y)$ → GD → boundary / sklearn → simulate last-look noise.


## 0. Packages


In [ ]:
import copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Toy FX blotter (Lab06 numbers, FX labels)

Six streamed quotes. Features are **desk scores**, not raw pips, so the Coursera gradient check still applies:

| pair | edge_score | urgency_score | lifted |
|------|------------|---------------|--------|
| USDCRC | 0.5 | 1.5 | 0 |
| EURUSD | 1.0 | 1.0 | 0 |
| GBPUSD | 1.5 | 0.5 | 0 |
| USDJPY | 3.0 | 0.5 | 1 |
| USDMXN | 2.0 | 2.0 | 1 |
| AUDUSD | 1.0 | 2.5 | 1 |

`edge_score` = how much better than mid the client is offered (scaled).  
`urgency_score` = invoice / payroll / travel deadline (scaled).  
`lifted` = 1 if the client converted.

### Task 1.1 — load the blotter


In [ ]:
# TODO: load data/fx_quote_blotter6.csv
# X_train = columns edge_score, urgency_score as ndarray (6,2)
# y_train = lifted as ndarray (6,)

# YOUR CODE HERE


print("X_train shape:", X_train.shape)
print("y_train:", y_train)
print("pairs:", df6["pair"].tolist())


### Task 1.2 — scatter passed vs lifted
Blue circles = passed, red crosses = lifted. Optional: annotate the pair.


In [ ]:
# TODO: scatter + optional pair labels
# YOUR CODE HERE


## 2. Logistic GD for P(lift)

Same updates as Lab06:

$$
w \leftarrow w - \alpha \frac{\partial J}{\partial w},\qquad
b \leftarrow b - \alpha \frac{\partial J}{\partial b}
$$

$$
\frac{\partial J}{\partial w} = \frac{1}{m}X^\top(f-y),\qquad
\frac{\partial J}{\partial b} = \frac{1}{m}\sum(f-y),\qquad
f=\sigma(Xw+b).
$$

### Task 2.1 — stable sigmoid


In [ ]:
def sigmoid(z):
    # YOUR CODE HERE
    pass

print(sigmoid(np.array([-100.0, 0.0, 100.0])))


### Task 2.2 — fill-cost $J$ (binary cross-entropy)


In [ ]:
def compute_cost_logistic(X, y, w, b):
    # YOUR CODE HERE
    pass


### Task 2.3 — loop gradient (`compute_gradient_logistic`)
For each quote: `err = σ(w·x+b) − y`, accumulate `err * x_j` and `err`, divide by $m$.


In [ ]:
def compute_gradient_logistic(X, y, w, b):
    """Return dj_db (scalar), dj_dw (n,)."""
    # YOUR CODE HERE
    pass


### Task 2.4 — published check
$w=(2,3)$, $b=1$ on this blotter must print

```
dj_db: 0.49861806546328574
dj_dw: [0.498333393278696, 0.49883942983996693]
```


In [ ]:
w_tmp = np.array([2., 3.])
b_tmp = 1.
dj_db_tmp, dj_dw_tmp = compute_gradient_logistic(X_train, y_train, w_tmp, b_tmp)
print("dj_db:", dj_db_tmp)
print("dj_dw:", dj_dw_tmp.tolist())


### Task 2.5 — vectorized alternate


In [ ]:
def compute_gradient_logistic_vec(X, y, w, b):
    # YOUR CODE HERE
    pass

dj_db_v, dj_dw_v = compute_gradient_logistic_vec(X_train, y_train, w_tmp, b_tmp)
print("vec match loop?", np.allclose(dj_dw_v, dj_dw_tmp) and np.isclose(dj_db_v, dj_db_tmp))


### Task 2.6 — batch GD


In [ ]:
def gradient_descent(X, y, w_in, b_in, alpha, num_iters):
    # YOUR CODE HERE
    pass


### Task 2.7 — train on the 6-quote blotter
Start at $w=0$, $b=0$, $\alpha=0.1$, 10 000 steps.


In [ ]:
# TODO: w_out, b_out, J_history = ...
# YOUR CODE HERE


### Task 2.8 — plot fill-cost $J$


In [ ]:
# TODO: linear + log10 J
# YOUR CODE HERE


### Task 2.9 — P(lift) field and 0.5 contour
Sales reads the black line as “quotes we expect to lift.” Pricing reads the colour as probability.


In [ ]:
# TODO: contourf P(lift) + 0.5 contour + scatter
# YOUR CODE HERE


## 3. Same blotter in scikit-learn (Lab07)

### Task 3.1 — default `LogisticRegression`
`fit`, `predict`, `score`. Default **L2** (`C=1`).


In [ ]:
from sklearn.linear_model import LogisticRegression

# YOUR CODE HERE


### Task 3.2 — unregularized vs L2 vs GD

On current sklearn use `LogisticRegression(C=np.inf, solver="lbfgs")` for an unregularized fit (`penalty=None` is deprecated).

Print coefficients and overlay three 0.5-contours. Training fill-rate can be 100% on six quotes while $w$ still disagrees — that is the industry lesson: **do not unit-test a pricing library against raw GD weights**.


In [ ]:
# TODO: C=np.inf + default L2 + overlay
# YOUR CODE HERE


## 4. One-feature retail spread

Load `data/fx_spread_1d.csv`. Feature for GD: **`tightness_bps`** (already $34 - \text{spread}$), target `converted`.

Fit GD. Report the spread (not tightness) at which $P(\text{convert})=0.5$.


In [ ]:
# TODO: 1-D GD on tightness; convert the cutoff back to spread_bps
# YOUR CODE HERE


## 5. More practice

### Task 5.1 — 60-quote stream
`data/fx_quote_practice.csv`. GD with $\alpha=0.3$, 2 000 steps, plus sklearn. Report fill-rate and sketch the boundary.


In [ ]:
# YOUR CODE HERE


### Task 5.2 — USD→CRC retail cart
`data/fx_usdcrc_markup.csv`: `usdcrc_markup_pct` vs `converted`.  
Fit 1-D GD. At what markup does $P(\text{convert})$ cross 50%? One sentence for a San José retail desk.


In [ ]:
# YOUR CODE HERE


### Task 5.3 — GD driven only by the vectorized gradient
Re-run the 6-quote fit. $w,b$ should match Task 2.7 within `1e-8`.


In [ ]:
# YOUR CODE HERE


## 6. Simulation — last-look noise and blotter size

Edit the knobs. Last-look / stale-quote noise flips a fraction of outcomes (the client saw a better rate elsewhere, or the quote aged).

1. $J$ vs iteration for several $\alpha$ on the 6-quote blotter.
2. Monte-Carlo train fill-rate vs number of quotes, with label flips.


In [ ]:
# --- editable parameters ---
ALPHAS = [0.01, 0.05, 0.1, 0.5, 1.0]
ITERS_ALPHA = 1500
SIZES = [20, 40, 80, 160]
N_REPS = 20
NOISE = 0.06          # last-look / stale-quote flip rate
ALPHA_MC = 0.3
ITERS_MC = 400
SEED = 11
# --------------------------

# YOUR CODE HERE


## 7. Audience rewrite

Same 6-quote result ($w$, $b$, fill-rate, sklearn L2 caveat), four voices (Jočys data-literacy + subject knowledge; McMurrey expert / technician / executive / nonspecialist):

1. Pricing quant / ML practitioner
2. Sales dealer / FX ops technician
3. Treasury / payments executive
4. Retail converter (travel money / remittance)


In [ ]:
practitioner = """..."""
dealer = """..."""
executive = """..."""
retail = """..."""


## 8. Done when
Gradient check matches, $J$ falls, 0.5-contour separates lifts from passes, sklearn comparison names L2, practice + USDCRC numbers exist, simulation reruns after you change a knob, four paragraphs keep the facts and change the vocabulary.

This is a **teaching blotter**, not a production auto-pricer and not FX trading advice.
